In [6]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import CharacterTextSplitter
import tiktoken

# 環境変数の読み込み
load_dotenv("../.env")
os.environ['OPENAI_API_KEY'] = os.environ['API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini"

# PDFファイルを読込
loader = DirectoryLoader('./data/pdf', glob="./*.pdf",   loader_cls=PyPDFLoader)
documents = loader.load()

# 言語モデルに合うトークナイザー名を取得
encoding_name = tiktoken.encoding_for_model(MODEL_NAME).name

# テキスト分割を作成
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(encoding_name)

# チャンクに分割
texts = text_splitter.split_documents(documents)

# エンベディングモデルの指定
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# インデックスの構築
db = Chroma.from_documents(texts, embedding_model)

# Retrieverの作成
retriever = db.as_retriever()

# 検索の実施
results = retriever.invoke("有給休暇の付与日数は？")

# プロンプトテンプレートの作成
prompt = ChatPromptTemplate.from_template("""提供されたコンテキストのみに基づいて次の質問に答えてください:

<コンテキスト>
{context}
</コンテキスト>

Question: {input}""")

chat_model = ChatOpenAI(model_name=MODEL_NAME)

chain = ({"context": retriever, "input": RunnablePassthrough()}
    | prompt
    | chat_model
    | StrOutputParser())

# チェーンの実行
response = chain.invoke("有給休暇の付与日数は？")

# 結果を表示
print(response)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


有給休暇の付与日数は、勤続年数に応じて以下のようになります：

- 0.5 年：10 日
- 1.5 年：11 日
- 2.5 年：12 日
- 3.5 年：14 日
- 4.5 年：16 日
- 5.5 年：18 日
- 6.5 年以上：20 日

初年度は入社から6ヶ月継続勤務し、全労働日の8割以上出勤した場合に10日間の有給休暇が付与され、その後は勤続年数に応じて増加します。
